<p class="lead">A dataset is not a fact; it is the output of a process, and every decision in that process is a choice you must be able to defend. This unit takes the raw ROCK1 download of Units 4 and 5 through a <strong>documented curation protocol</strong> written as tested R functions, in which every excluded record carries a reason code and the exclusion counts are reproduced exactly against a reference implementation. It then asks the three questions that decide whether the resulting table can support a model at all: how much experimental noise does it contain, how arbitrary are the class labels we are about to impose, and how concentrated is the data in a few chemical series and a few papers. The last section leaves chemistry for the legal and ethical obligations that attach to human-derived data, and the unit ends with a <strong>data card</strong> — the document that travels with the dataset.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad convierte una descarga cruda en una tabla apta para modelar, de forma documentada y verificable. Implementamos el protocolo de curación como funciones de R con pruebas <code>testthat</code>: cada registro excluido lleva un código de razón, se registra el recuento por paso y se dibuja el diagrama de flujo; al final reproducimos exactamente la tabla de referencia del curso, con <code>stopifnot()</code> sobre el recuento de 4.280 compuestos. Después estimamos la <em>incertidumbre experimental</em> a partir de las réplicas del propio conjunto y la traducimos en un techo para el rendimiento de cualquier modelo. Analizamos la sensibilidad al umbral de actividad (pActivity ≥ 6,0 / 6,5 / 7,0) sobre el balance de clases y la composición de andamiajes, y diagnosticamos los sesgos: concentración en pocas series químicas y pocos documentos (curvas de Lorenz y coeficiente de Gini), andamiajes únicos y la fuga de información de una partición aleatoria. La última parte trata el manejo responsable de datos derivados de personas — anonimización, k-anonimato, l-diversidad, minimización, comité de ética, RGPD y marco chileno — usando solo el esquema de metadatos de una serie pública de expresión génica, sin reidentificar nada. Cerramos con la ficha de datos del conjunto final.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Apply a documented curation protocol to chemical bioactivity data — structure standardisation, duplicates, censored values, assay heterogeneity, activity thresholds — and report what was removed and why.</li>
<li>Write the protocol as a tested function pipeline with reason-coded exclusions, a curation log and a flow diagram, and verify it against an independent implementation.</li>
<li>Estimate the experimental uncertainty of a public dataset from its own replicates and convert it into a ceiling on achievable model performance.</li>
<li>Quantify how the choice of activity threshold changes class balance and scaffold composition.</li>
<li>Diagnose dataset biases — analogue bias, source bias, singleton scaffolds, class imbalance — and state their effect on validation.</li>
<li>Handle clinical and human-derived data lawfully and ethically: de-identification, k-anonymity, data minimisation, ethics-committee approval and the applicable data-protection framework.</li>
<li>Produce a data card documenting the final table.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Read linearly, running each chunk. Green <strong>domain-context boxes</strong> define every chemical, biological and legal term on first use. In a fresh Google Colab R runtime, run once in a cell of your own: <code>install.packages(c("tidyverse","patchwork","testthat","jsonlite"))</code>. Section 8 uses the metadata of a public gene-expression series: we inspect its <em>schema</em> only and never attempt to link any record to a person — the point of the section is to show why that restraint is a design requirement rather than a courtesy. This unit follows Units 4 and 5 and feeds Units 11–12 (the modelling table and its validation) and Unit 23 (reporting).</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(dplyr); library(tidyr); library(stringr); library(ggplot2)
  library(patchwork); library(testthat); library(jsonlite); library(digest)
})
set.seed(0)
options(width = 120, digits = 6)
invisible(Sys.setlocale("LC_CTYPE", "en_US.UTF-8"))

PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 9.5),
                panel.grid.minor = element_blank(),
                axis.line = element_line(colour = "grey40", linewidth = 0.3),
                legend.key.size = unit(0.3, "cm")))

REPO <- "DanielBustosG/DanielBustosG.github.io"
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO,
                   "/main/DataScience/data/")
data_path <- function(name) {
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE)
  }
  dest
}
load_data <- function(name, ...)
  read.csv(data_path(name), stringsAsFactors = FALSE, na.strings = c("", "NA"), ...)

cat("R", as.character(getRversion()), "| dplyr", as.character(packageVersion("dplyr")),
    "| testthat", as.character(packageVersion("testthat")),
    "| jsonlite", as.character(packageVersion("jsonlite")), "\n")



## 1. What "quality" means for a dataset

A dataset has no intrinsic quality; it has *fitness for a stated purpose*. The ROCK1 table is fit to answer "which chemical series were explored against this kinase and how potent were they?" It is much less fit to answer "what is the potency of an arbitrary new molecule?", and the gap between those two statements is what this unit measures.

Four separate things are usually bundled under "data quality", and they need separate treatment.

1. **Correctness.** Are the records right? Structures mis-drawn, exponents mis-typed, units mis-read. Audits of public chemistry databases consistently find error rates of a few per cent, concentrated in exactly these places.<sup class="c"><a href="#r6">[6]</a></sup><sup class="c"><a href="#r7">[7]</a></sup><sup class="c"><a href="#r8">[8]</a></sup>
2. **Precision.** How reproducible is a correct measurement? This is the experimental uncertainty of Section 4, and it is a hard ceiling on what any model can achieve.
3. **Representativeness.** Does the dataset look like the population you will apply the model to? This is the bias question of Section 7, and it is the one most often ignored.
4. **Documentation.** Can a reader reconstruct what you did and decide whether to trust it? This is the data card of Section 9, and it is what makes the other three checkable.<sup class="c"><a href="#r21">[21]</a></sup>

The published curation protocols for chemical data — Fourches, Muratov and Tropsha's "Trust, but verify" and its sequel<sup class="c"><a href="#r1">[1]</a></sup><sup class="c"><a href="#r2">[2]</a></sup> — address the first and fourth; this unit implements them and then adds the second and third, which are what decide whether a model built on the table will generalise.

<div class="box bridge"><h4>Domain context — what can go wrong in a bioactivity record</h4>
<p>Each record says that a <strong>compound</strong> (a small molecule) showed a certain potency against the enzyme ROCK1 in a particular <strong>assay</strong> (one experimental protocol). Five things routinely go wrong. The structure may be deposited as a <strong>salt</strong> or a different <strong>tautomer</strong> — the same molecule drawn with a hydrogen atom in a different place — so two records that should be one are two. The potency may be a <strong>censored</strong> value ("&gt; 10 µM") because the compound was too weak to measure. Different assays measure different quantities: an <strong>IC50</strong> depends on the experimental conditions, a <strong>Ki</strong> does not, and mixing them adds noise. The compound may be a <strong>frequent hitter</strong> or <strong>PAINS</strong> (pan-assay interference compound), a molecule that appears active in many unrelated assays because it aggregates, fluoresces or reacts rather than because it binds the target.<sup class="c"><a href="#r12">[12]</a></sup> And the set of compounds in the database is not a sample of chemical space but the result of decades of <strong>optimisation campaigns</strong>, so it is dominated by a few families of close analogues — <strong>analogue bias</strong>.</p></div>

## 2. The curation protocol as tested functions

A protocol is only reproducible if it is code, and only auditable if every exclusion is recorded with a reason. The pattern below is worth copying into every project: a small list-based *curator* carried through a pipeline, where each `keep_if()` applies one named filter, records how many rows it removed and why, and stores the removed rows for inspection. Writing it as plain functions on a plain list — rather than as a reference class — keeps it pipeable and, more importantly, testable: each function takes a value and returns a value.



In [ ]:
new_curator <- function(df, id_col = "molecule_chembl_id") {
  cur <- list(df = df, id_col = id_col, log = list(), excluded = list())
  record(cur, "E00", "raw records as delivered by the ChEMBL API", 0)
}
record <- function(cur, code, description, n_removed, note = "") {
  cur$log[[length(cur$log) + 1]] <- data.frame(
    step = length(cur$log), code = code, description = description,
    n_removed = n_removed, n_rows = nrow(cur$df),
    n_molecules = dplyr::n_distinct(cur$df[[cur$id_col]]), note = note)
  cur
}
keep_if <- function(cur, code, description, mask, note = "") {
  mask <- !is.na(mask) & mask                       # NA is never "keep": say so once, here
  if (any(!mask)) cur$excluded[[code]] <- cur$df[!mask, , drop = FALSE]
  cur$df <- cur$df[mask, , drop = FALSE]
  record(cur, code, description, sum(!mask), note)
}
transform_with <- function(cur, code, description, fn, note = "") {
  cur$df <- fn(cur$df)
  record(cur, code, description, 0, note)
}
log_frame <- function(cur) do.call(rbind, cur$log)

raw <- load_data("rock1_chembl_raw.csv")
cat("raw file:", nrow(raw), "rows x", ncol(raw), "columns\n")
cat("SHA-256 of the file:", substr(digest(data_path("rock1_chembl_raw.csv"),
                                         algo = "sha256", file = TRUE), 1, 16), "...\n")
cat("curator defined; the protocol follows.\n")



The `mask <- !is.na(mask) & mask` line in `keep_if()` is the one piece of defensive R worth pointing at. A logical vector in R has three values, not two, and `df[c(TRUE, NA, FALSE), ]` returns a row of `NA`s rather than dropping the row — a silent corruption that has ended more than one analysis. Deciding once, in the framework, that a missing predicate means "exclude" is better than remembering it at thirteen call sites.

Before trusting the framework with a real protocol, test it. `testthat` runs the same tests interactively and in a package, so there is no reason not to write them.



In [ ]:
toy <- data.frame(molecule_chembl_id = c("A", "B", "B", "C"),
                  value = c(1, NA, 3, 4))
test_that("keep_if removes exactly the rows failing the mask", {
  cur <- new_curator(toy) |> keep_if("T01", "value present", !is.na(toy$value))
  expect_equal(nrow(cur$df), 3)
  expect_equal(log_frame(cur)$n_removed, c(0, 1))
})
test_that("an NA in the mask excludes the row rather than corrupting it", {
  cur <- new_curator(toy) |> keep_if("T02", "value > 2", toy$value > 2)
  expect_equal(nrow(cur$df), 2)
  expect_false(any(is.na(cur$df$molecule_chembl_id)))
})
test_that("the log records rows, molecules and the excluded frame", {
  cur <- new_curator(toy) |> keep_if("T03", "not B", toy$molecule_chembl_id != "B")
  lf <- log_frame(cur)
  expect_equal(lf$n_rows, c(4, 2)); expect_equal(lf$n_molecules, c(3, 2))
  expect_equal(nrow(cur$excluded[["T03"]]), 2)
})
test_that("transform_with changes columns without removing rows", {
  cur <- new_curator(toy) |> transform_with("T04", "double", \(x) transform(x, value = value * 2))
  expect_equal(nrow(cur$df), 4); expect_equal(log_frame(cur)$n_removed, c(0, 0))
})
cat("all four curator tests pass\n")



Four tests, four properties: the filter removes the right number of rows, a missing predicate excludes rather than corrupts, the log counts both rows and molecules and keeps the excluded frame, and a transformation removes nothing. They run in under a second and they are the reason the protocol below can be trusted without being read line by line.

<div class="exercise"><h4>Exercise 1</h4>
<p>Before running the next chunk, predict which step will remove the most records. Write the number down. Most people guess the duplicate step or the validity flags; the answer is in the log, and the gap between your guess and it is the reason protocols are logged rather than remembered.</p></div>

<div class="box"><h4>Substitution in the R edition — structure standardisation</h4>
<p>The Python edition standardises every deposited SMILES with RDKit (<code>Normalizer</code>, <code>LargestFragmentChooser</code>, <code>Uncharger</code>), computes the InChIKey and counts heavy atoms inside the notebook. R has no RDKit binding in this environment, so steps E07 and E08 of the protocol are <strong>merged into one step that joins the frozen result</strong> of that standardisation: <code>rock1_chembl_curated.csv</code> lists, in its <code>chembl_ids</code> column, every ChEMBL identifier that collapsed into each standardised parent, and only structures that already passed the 3–70 heavy-atom window appear there. The structures themselves ship as <code>data/rock1_curated.sdf.gz</code> and the generation recipe is in <code>data/manifest_fingerprints.json</code>, so a reader with ChemmineR or rcdk can redo the chemistry. The consequence for the log is that the R edition reports one merged exclusion step where the Python edition reports two; the record counts are identical.</p></div>



In [ ]:
UNITS_TO_NM <- c("nM" = 1, "uM" = 1e3, "\u00b5M" = 1e3, "mM" = 1e6, "M" = 1e9,
                 "pM" = 1e-3, "fM" = 1e-6, "nmol/L" = 1, "umol/L" = 1e3)
curated <- load_data("rock1_chembl_curated.csv") |>
  mutate(censored = censored == "True", wide_spread = wide_spread == "True")
key_map <- curated |>
  select(inchikey, std_smiles, chembl_ids, heavy = HeavyAtoms, scaffold_smiles) |>
  separate_rows(chembl_ids, sep = "\\|") |>
  rename(molecule_chembl_id = chembl_ids)

cur <- new_curator(raw)
cur <- keep_if(cur, "E01", "no deposited structure", !is.na(cur$df$canonical_smiles))
cur <- keep_if(cur, "E02", "target organism is not human",
               cur$df$target_organism == "Homo sapiens")
flag_tab <- table(cur$df$data_validity_comment)
cur <- keep_if(cur, "E03", "ChEMBL data_validity_comment flag",
               is.na(cur$df$data_validity_comment),
               note = paste(names(flag_tab), flag_tab, collapse = "; "))
cur <- keep_if(cur, "E04", "no numeric standard_value", !is.na(cur$df$standard_value))
cur <- transform_with(cur, "T05", "harmonise units to nM",
                      \(x) mutate(x, value_nM = unname(UNITS_TO_NM[standard_units]) * standard_value))
cur <- keep_if(cur, "E05", "unit not convertible to nM", !is.na(cur$df$value_nM),
               note = paste(unique(cur$df$standard_units[is.na(cur$df$value_nM)]), collapse = ","))
cur <- keep_if(cur, "E06", "non-positive or absurd concentration (<=0 or >1e8 nM)",
               cur$df$value_nM >= 1e-12 & cur$df$value_nM <= 1e8)
cur <- transform_with(cur, "T07", "attach the standardised parent (frozen RDKit map)",
                      \(x) left_join(x, key_map, by = "molecule_chembl_id",
                                     relationship = "many-to-one"))
cur <- keep_if(cur, "E07", "no standardised parent within 3-70 heavy atoms",
               !is.na(cur$df$inchikey) & cur$df$heavy >= 3 & cur$df$heavy <= 70)
cur <- transform_with(cur, "T09", "derive pActivity and the censoring flag",
                      \(x) mutate(x, pActivity = 9 - log10(value_nM),
                                  censored = coalesce(standard_relation, "=") != "="))
measurements <- cur$df
print(log_frame(cur)[c("step", "code", "description", "n_removed", "n_rows", "n_molecules")],
      row.names = FALSE)



<p class="caption"><b>Table (Section 2).</b> The curation log. Forty-four of 5,089 records are excluded in total — 3 without a structure, 13 carrying ChEMBL's own "Outside typical range" flag, 26 with no numeric value, and 2 whose standardised parent falls outside the heavy-atom window. No record is lost to unit conversion, because ChEMBL's <code>standard_units</code> column is already nanomolar throughout; the conversion table matters for the <em>reported</em> column that Unit 4 harmonised by hand. The protocol is deliberately conservative: it removes only records that are unusable, not records that are inconvenient.</p>

The reason codes are not decoration. Because every exclusion was stored, any step can be audited by looking at what it threw away.



In [ ]:
for (code in names(cur$excluded)) {
  fr <- cur$excluded[[code]]
  cols <- intersect(c("molecule_chembl_id", "type", "value", "units", "standard_value",
                      "data_validity_comment", "activity_comment"), names(fr))
  cat(sprintf("[%s] %d record(s)\n", code, nrow(fr)))
  print(head(fr[cols], 3), row.names = FALSE)
  cat("\n")
}
cat("E03 removed the flagged rows; their values in nM:\n")
print(sort(cur$excluded[["E03"]]$standard_value))



The thirteen flagged records are instructive. Twelve are potencies between 120,000 and 30,000,000 nM — that is, between 120 µM and 30 mM, concentrations at which nothing specific is being measured because the compound would precipitate first. The thirteenth is an IC50 of exactly **0 nM**, which is not a weak measurement but an impossible one: its logarithm is infinite, and had it survived it would have propagated silently into every mean, standard deviation and model fit downstream. In R it would have produced `-Inf` rather than an error, and `-Inf` passes quietly through `mean()`, `sd()` and `lm()` until something far away reports `NaN`. ChEMBL's curators flagged all thirteen; our protocol removes them and says so.

<div class="box bridge"><h4>Domain context — why structure standardisation comes before de-duplication</h4>
<p>The same molecule can be deposited many ways: as a <strong>salt</strong> (with a counter-ion such as chloride attached by a dot in the SMILES string), as a <strong>hydrate</strong>, in a charged form, or with a functional group drawn in a non-standard but equivalent way. <strong>Standardisation</strong> applies a fixed sequence — normalise the drawing, keep the largest fragment, neutralise charges — so that all of these collapse to one canonical <em>parent</em> structure, which is then hashed into an <strong>InChIKey</strong>.<sup class="c"><a href="#r9">[9]</a></sup><sup class="c"><a href="#r10">[10]</a></sup> De-duplicating before standardising leaves the salt forms as separate compounds; in this dataset that would split ripasudil into three. Every published curation protocol therefore puts standardisation first.<sup class="c"><a href="#r1">[1]</a></sup></p></div>

## 3. Aggregation and verification against the reference

The remaining step is the measurement-to-compound aggregation of Unit 4. The descriptors and the Bemis–Murcko scaffold that the modelling units need were computed once with RDKit and ship as columns of the frozen table, so here they are joined rather than recomputed. Then comes the check that matters: does our table equal the reference table the laboratory ships with the course?



In [ ]:
comp <- measurements |>
  group_by(inchikey) |>
  mutate(use = if (any(!censored)) !censored else rep(TRUE, n())) |>
  summarise(std_smiles     = first(std_smiles),
            pref_name      = if (all(is.na(pref_name))) NA_character_ else na.omit(pref_name)[1],
            max_phase      = suppressWarnings(max(max_phase, na.rm = TRUE)),
            pActivity_min  = min(pActivity[use]),
            pActivity_max  = max(pActivity[use]),
            pActivity      = median(pActivity[use]),
            n_measurements = n(),
            n_exact        = sum(!censored),
            n_IC50         = sum(standard_type == "IC50"),
            n_Ki           = sum(standard_type == "Ki"),
            n_assays       = n_distinct(assay_chembl_id),
            n_documents    = n_distinct(document_chembl_id),
            scaffold_smiles = first(scaffold_smiles),
            .groups = "drop") |>
  mutate(max_phase = ifelse(is.finite(max_phase), max_phase, NA_real_),
         censored = n_exact == 0,
         pActivity_range = pActivity_max - pActivity_min,
         wide_spread = pActivity_range > 1)
comp <- comp |>
  left_join(curated |> select(inchikey, MolWt, MolLogP, TPSA, HBD, HBA, RotB,
                              AromaticRings, RingCount, HeavyAtoms, FractionCSP3, QED),
            by = "inchikey", relationship = "one-to-one")
cur <- record(cur, "A10", "aggregate replicate measurements per InChIKey (median)",
              nrow(measurements) - nrow(comp), note = "compound level")
cat("compound-level table:", nrow(comp), "rows x", ncol(comp), "columns\n")
print(as.data.frame(comp |> filter(!is.na(pref_name)) |> slice_head(n = 3) |>
  select(pref_name, pActivity, n_measurements, n_assays, MolWt, QED) |>
  mutate(across(where(is.numeric), \(x) round(x, 2)))))


In [ ]:
ref <- curated
ref_log <- load_data("rock1_curation_log.csv")
cat("our compound count:", nrow(comp), " | reference compound count:", nrow(ref), "\n")
stopifnot("compound count must equal the reference protocol" = nrow(comp) == 4280)
stopifnot("compound count must equal the reference table" = nrow(comp) == nrow(ref))
stopifnot("the same structures, not just the same number" =
            setequal(comp$inchikey, ref$inchikey))
mg <- comp |>
  select(inchikey, pActivity_ours = pActivity, censored_ours = censored,
         scaffold_ours = scaffold_smiles) |>
  inner_join(ref |> select(inchikey, pActivity_ref = pActivity, censored_ref = censored,
                           scaffold_ref = scaffold_smiles),
             by = "inchikey", relationship = "one-to-one")
dmax <- max(abs(mg$pActivity_ours - mg$pActivity_ref))
cat("identical censoring flags:", all(mg$censored_ours == mg$censored_ref), "\n")
cat("identical Murcko scaffolds:", all(mg$scaffold_ours == mg$scaffold_ref), "\n")
cat(sprintf("max |pActivity difference|: %.4f log units\n", dmax))
stopifnot(dmax < 1e-3)
cat("\nreference log (final row) vs ours:\n")
cat("  reference:", ref_log$description[nrow(ref_log)], "->", ref_log$n_rows[nrow(ref_log)], "rows\n")
cat("  ours     :", cur$log[[length(cur$log)]]$description, "->", nrow(comp), "rows\n")
cat("\nmeasurements surviving curation: ours", nrow(measurements),
    "| reference", ref_log$n_rows[ref_log$step == 9], "\n")



Our independently written protocol produces **4,280 compounds**, the same 4,280 as the reference table, with identical censoring flags, identical Murcko scaffolds and potencies agreeing to better than 0.001 log units; at the measurement level both keep **5,045** of the original 5,089 records. The `stopifnot()` on the compound count is deliberately written as a literal 4,280 as well as a comparison with the file, so that a silent change in either the protocol or the shipped table breaks the build rather than the conclusions. That equality is the licence to use the table: three implementations of the same written protocol — the reference script, the Python edition and this one — agreeing exactly, is the cheapest form of reproducibility there is.<sup class="c"><a href="#r24">[24]</a></sup>

The log deserves a picture, because a flow diagram is what a referee asks for.



In [ ]:
lg <- log_frame(cur) |> filter(str_starts(code, "E") | code == "A10") |>
  mutate(order = row_number(),
         label = ifelse(code == "A10",
                        sprintf("%s  %s\ncompounds: %s", code, description,
                                format(nrow(comp), big.mark = ",")),
                        sprintf("%s  %s\nrecords: %s    compounds: %s", code, description,
                                format(n_rows, big.mark = ","),
                                format(n_molecules, big.mark = ","))),
         y = -order)
print(ggplot(lg) +
  geom_tile(aes(0, y, fill = code == "A10"), width = 1.9, height = 0.62, colour = "grey40",
            linewidth = 0.3) +
  geom_text(aes(0, y, label = label), size = 2.3, lineheight = 1.05) +
  geom_segment(data = filter(lg, order < max(order)),
               aes(x = 0, xend = 0, y = y - 0.31, yend = y - 0.69),
               arrow = arrow(length = unit(0.07, "cm"), type = "closed"),
               colour = "grey45", linewidth = 0.3) +
  geom_segment(data = filter(lg, n_removed > 0),
               aes(x = 0.98, xend = 1.25, y = y, yend = y),
               arrow = arrow(length = unit(0.07, "cm"), type = "closed"),
               colour = unname(PAL["orange"]), linewidth = 0.3) +
  geom_text(data = filter(lg, n_removed > 0),
            aes(1.3, y, label = paste0("\u2212", n_removed,
                                       ifelse(code == "A10", " rows collapsed", " excluded"))),
            hjust = 0, size = 2.2, colour = unname(PAL["orange"])) +
  scale_fill_manual(values = c(`FALSE` = "#dce6f1", `TRUE` = "#d7e6c8"), guide = "none") +
  coord_cartesian(xlim = c(-1.1, 2.3)) +
  labs(title = "ROCK1 curation protocol: records in, reasons out") +
  theme_void(base_size = 9) +
  theme(plot.title = element_text(face = "bold", size = 10, hjust = 0.5)))
total_excluded <- sum(log_frame(cur)$n_removed[log_frame(cur)$code != "A10"])
cat("total records excluded:", total_excluded, "of", nrow(raw),
    sprintf("(%.1f%%)\n", 100 * total_excluded / nrow(raw)))



<p class="caption"><b>Figure 1.</b> The curation flow diagram, generated from the log object rather than drawn by hand — so it cannot drift out of step with the code. Each box is one documented decision, each orange arrow an exclusion with its count. Forty-four records (0.9%) are excluded in total; the final box is the aggregation, where the remaining 765 rows are not lost but collapsed into the compounds that carry them. A figure of this shape, with the reason codes, is what a methods section should contain; in CONSORT-style reporting for clinical studies it is mandatory, and there is no good reason for cheminformatics to hold itself to a lower standard.<sup class="c"><a href="#r2">[2]</a></sup></p>

## 4. How noisy is this dataset? Ask the data

Nothing in the protocol so far says how *precise* the surviving numbers are, and precision is the property that decides what a model can achieve. The literature answer is well known: when the same compound–target pair is measured in two different laboratories, the two values differ by something of the order of half a log unit. Kramer and colleagues established this for heterogeneous public Ki data,<sup class="c"><a href="#r3">[3]</a></sup> Kalliokoski and colleagues for IC50 data pooled across sources,<sup class="c"><a href="#r4">[4]</a></sup> and Landrum and Riniker revisited the question with a decade more data and reached the same conclusion.<sup class="c"><a href="#r5">[5]</a></sup> We do not have to take that on trust, because our own dataset contains the replicates needed to estimate it.

The estimator is simple. For every compound measured in **two different documents**, take the mean pActivity reported by each document and form the difference. If each document's value is the true value plus independent noise of standard deviation \( \sigma \), then each difference has standard deviation \( \sigma\sqrt{2} \), so

\[ \hat{\sigma} \;=\; \sqrt{\tfrac{1}{2}\,\overline{\Delta^2}}. \]

The restriction to *different documents* matters: two values from the same paper are not independent measurements, they are the same experiment reported twice.



In [ ]:
exact <- measurements |> filter(!censored)

interdoc_pairs <- function(df) {
  per_doc <- df |>
    group_by(inchikey, document_chembl_id) |>
    summarise(m = mean(pActivity), .groups = "drop_last") |>
    summarise(n_docs = n(), vals = list(m), .groups = "drop") |>
    filter(n_docs > 1)
  unlist(lapply(per_doc$vals, \(v) as.numeric(dist(v))))     # all pairwise |differences|
}
unc <- lapply(list(`all types pooled` = exact,
                   `IC50 only` = filter(exact, standard_type == "IC50"),
                   `Ki only`   = filter(exact, standard_type == "Ki")),
  function(sub) {
    dd <- interdoc_pairs(sub)
    data.frame(n_pairs = length(dd), mean_abs_diff = mean(dd), median_abs = median(dd),
               rms_diff = sqrt(mean(dd^2)), sigma = sqrt(mean(dd^2) / 2),
               frac_gt_1_log = mean(dd > 1))
  })
unc <- cbind(subset = names(unc), do.call(rbind, unc))
print(unc, row.names = FALSE, digits = 3)
sigma_hat <- unc$sigma[1]
cat(sprintf("\nestimated experimental uncertainty of one measurement: sigma = %.3f log units\n",
            sigma_hat))
cat(sprintf("i.e. a 68%% interval of -%.0f%% to +%.0f%% in concentration\n",
            100 * (1 - 10^-sigma_hat), 100 * (10^sigma_hat - 1)))



Seven hundred and eighty inter-document pairs give a root-mean-square difference of 0.716 log units and therefore an estimated per-measurement uncertainty of **σ ≈ 0.506 log units** — a factor of about 3.2 in concentration at one standard deviation. Split by measurement type, IC50 data give σ ≈ 0.475 and Ki data σ ≈ 0.416, both consistent with the published estimates of roughly half a log unit.<sup class="c"><a href="#r3">[3]</a></sup><sup class="c"><a href="#r4">[4]</a></sup> That our number, computed on one target from one database, lands where two large independent studies landed is reassuring about the method and sobering about the data. Note also the gap between the mean absolute difference (0.416) and the root mean square (0.716): the distribution is sharply peaked at zero with a long tail, so the two summaries describe different parts of it, and it is the squared version that feeds the variance budget of Section 5.

A second comparison is available because 26 compounds in this set have both an IC50 and a Ki.



In [ ]:
both <- exact |>
  group_by(inchikey) |>
  filter(any(standard_type == "IC50"), any(standard_type == "Ki")) |>
  summarise(d = median(pActivity[standard_type == "IC50"]) -
                median(pActivity[standard_type == "Ki"]), .groups = "drop")
cat("compounds with both IC50 and Ki:", nrow(both), "\n")
cat(sprintf("pIC50 - pKi:  mean %+.3f   median %+.3f   sd %.3f   range %+.2f to %+.2f\n",
            mean(both$d), median(both$d), sd(both$d), min(both$d), max(both$d)))
dd <- interdoc_pairs(exact)
rms <- sqrt(mean(dd^2))
n1 <- ggplot(data.frame(d = dd), aes(d)) +
  geom_histogram(binwidth = 0.1, boundary = 0, fill = PAL["blue"]) +
  geom_vline(xintercept = rms, colour = PAL["orange"], linewidth = 0.6) +
  annotate("text", x = rms, y = Inf, vjust = 1.5, hjust = -0.1, size = 2.4,
           colour = unname(PAL["orange"]), label = sprintf("RMS = %.2f", rms)) +
  labs(x = "|difference| between two documents (log units)", y = "compound pairs",
       title = paste0("Inter-document replicates (n=", length(dd), ")"))
n2 <- ggplot(unc, aes(reorder(subset, -sigma), sigma)) +
  geom_col(fill = PAL["green"], width = 0.6) +
  geom_hline(yintercept = 0.5, linetype = "dashed", colour = PAL["orange"], linewidth = 0.4) +
  geom_text(aes(label = sprintf("%.3f", sigma)), vjust = -0.4, size = 2.4) +
  labs(x = NULL, y = "estimated sigma (log units)", title = "Uncertainty by measurement type") +
  theme(axis.text.x = element_text(angle = 15, hjust = 1, size = 6.5))
n3 <- ggplot(both, aes(d)) +
  geom_histogram(binwidth = 0.25, boundary = 0, fill = PAL["orange"]) +
  geom_vline(xintercept = 0, colour = "grey30", linewidth = 0.3) +
  geom_vline(xintercept = mean(both$d), linetype = "dashed", linewidth = 0.4) +
  labs(x = "pIC50 - pKi for the same compound", y = "compounds",
       title = paste0("IC50 vs Ki (n=", nrow(both), ")"))
print(n1 | n2 | n3)



<p class="caption"><b>Figure 2.</b> Left: the distribution of absolute differences between two documents reporting the same compound. It is sharply peaked at zero — the median difference is 0.166 log units, so most repeated measurements agree well — with a long tail reaching beyond three log units, and the root-mean-square of 0.716 is dominated by that tail; 13.5% of the pairs disagree by more than a log unit. Middle: the implied per-measurement σ for the pooled data (0.506) and by type (IC50 0.475, Ki 0.416), against the ≈0.5 log-unit figure from the literature. Right: for the 26 compounds measured both ways, pIC50 minus pKi has a mean of −0.87 and a standard deviation of 0.80 — the IC50 is systematically about one log unit <em>weaker</em> than the Ki for the same molecule, which is what the Cheng–Prusoff relation predicts when an assay is run at an ATP concentration near the enzyme's K<sub>M</sub>. Pooling IC50 and Ki values into one target column, as almost every public dataset does, therefore adds both a bias and a variance.</p>

## 5. The noise ceiling: what the data allow a model to achieve

The uncertainty estimate is not a curiosity; it is a budget. Write each observed value as a true value plus independent noise, \( y = t + \varepsilon \) with \( \mathrm{Var}(\varepsilon) = \sigma^2 \). The observed variance splits as

\[ \sigma_y^2 \;=\; \sigma_t^2 + \sigma^2, \]

so even a *perfect* model — one that returns the true value \( t \) exactly — cannot explain the \( \sigma^2 \) part. Its coefficient of determination against the observed values is capped at

\[ R^2_{\max} \;=\; 1 - \frac{\sigma^2}{\sigma_y^2}, \]

and its root-mean-square error cannot fall below \( \sigma \). This quantity is the **reliability** of the measurement, and it is the number to compare a model's reported performance against — not zero, and not one.

<h4>Worked example by hand</h4>

Both calculations are small enough to do on paper, which is the best way to see that neither hides anything.

*The uncertainty.* Suppose one compound is reported by three documents at pActivity 7.10, 6.40 and 6.85. The three pairwise differences are |7.10 − 6.40| = 0.70, |7.10 − 6.85| = 0.25 and |6.85 − 6.40| = 0.45. Their squares are 0.4900, 0.0625 and 0.2025, whose mean is 0.7550/3 = 0.2517, so the root-mean-square difference is √0.2517 = 0.5017 and the implied per-measurement uncertainty is 0.5017/√2 = **0.355** log units. The chunk above does exactly this over all 780 pairs in the dataset and obtains 0.506.

*The ceiling.* With σ = 0.506 the noise variance is 0.506² = 0.2560; the observed standard deviation of pActivity over the compounds with exact values is 1.1085, so the observed variance is 1.2288. Therefore

| quantity | arithmetic | value |
|---|---|---|
| noise variance σ² | 0.506 × 0.506 | 0.2560 |
| observed variance σ²<sub>y</sub> | 1.1085 × 1.1085 | 1.2288 |
| true-signal variance | 1.2288 − 0.2560 | 0.9728 |
| R²<sub>max</sub> | 1 − 0.2560 / 1.2288 | **0.7917** |
| RMSE floor | σ | **0.506** |

The chunk below prints 0.792 and 0.506, confirming the hand calculation to three decimals.



In [ ]:
sy <- sd(comp$pActivity[!comp$censored])
var_obs <- sy^2; var_noise <- sigma_hat^2; var_true <- var_obs - var_noise
cat(sprintf("observed sd of pActivity      : %.3f  (variance %.3f)\n", sy, var_obs))
cat(sprintf("estimated measurement noise   : %.3f  (variance %.3f)\n", sigma_hat, var_noise))
cat(sprintf("implied true-signal variance  : %.3f\n", var_true))
cat(sprintf("reliability = R2 ceiling      : %.3f\n", var_true / var_obs))
cat(sprintf("RMSE floor                    : %.3f log units\n", sigma_hat))

set.seed(0)
n <- sum(!comp$censored)
true <- rnorm(n, mean(comp$pActivity[!comp$censored]), sqrt(var_true))
obs1 <- true + rnorm(n, 0, sigma_hat)
obs2 <- true + rnorm(n, 0, sigma_hat)
r2 <- function(y, pred) 1 - sum((y - pred)^2) / sum((y - mean(y))^2)
cat(sprintf("\nsimulated: a perfect model scored against observed values -> R2 = %.3f, RMSE = %.3f\n",
            r2(obs1, true), sqrt(mean((obs1 - true)^2))))
cat(sprintf("simulated: one replicate used to predict another           -> R2 = %.3f\n",
            r2(obs2, obs1)))
acc <- vapply(c(6.0, 6.5, 7.0), \(t) mean((obs1 >= t) == (true >= t)), numeric(1))
cat("\nclassification ceiling (a perfect model scored against a noisy label):\n")
print(setNames(round(acc, 3), paste0("pActivity >= ", c(6.0, 6.5, 7.0))))



The numbers are blunt. With σ = 0.506 and an observed spread of 1.108 log units, the reliability of this dataset is **0.792**: no regression model, however good, can exceed R² ≈ 0.79 or fall below an RMSE of about 0.51 log units when scored against these labels. A paper reporting R² = 0.95 on a public bioactivity set of this kind is not reporting a better model; it is reporting a leak. And on the classification side, a perfect model assigning the true class is scored against a noisy label, so even it agrees with the label only 84–87% of the time at the three thresholds.

Two caveats, both important. The estimate assumes the between-document differences are pure noise, when some of them are real differences between assays — which makes σ an *upper* bound on the random error and the ceiling a lower bound on achievable performance. And it is an average: a compound measured 20 times has a much better-determined median than one measured once, so the ceiling is not uniform across the table.



In [ ]:
sig <- seq(0.05, 0.9, length.out = 120)
c1 <- ggplot(data.frame(sig, ceiling = 1 - sig^2 / var_obs), aes(sig, ceiling)) +
  geom_line(colour = PAL["blue"], linewidth = 0.7) +
  geom_vline(xintercept = sigma_hat, linetype = "dashed", colour = PAL["orange"],
             linewidth = 0.4) +
  annotate("text", x = sigma_hat + 0.03, y = 1 - sigma_hat^2 / var_obs + 0.14, hjust = 0,
           size = 2.4, colour = unname(PAL["orange"]),
           label = sprintf("our estimate\nsigma = %.2f -> R2 <= %.2f",
                           sigma_hat, 1 - sigma_hat^2 / var_obs)) +
  ylim(0, 1) +
  labs(x = "assumed measurement sigma (log units)", y = "maximum achievable R2",
       title = "The noise ceiling")
c2 <- ggplot(data.frame(true, obs1), aes(true, obs1)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", colour = "grey30",
              linewidth = 0.3) +
  geom_point(size = 0.4, alpha = 0.25, colour = PAL["blue"]) +
  coord_cartesian(xlim = c(2.5, 10.5), ylim = c(2.5, 10.5)) +
  labs(x = "true value (simulated)", y = "observed value = true + noise",
       title = sprintf("A perfect model's scatter: R2 = %.2f", r2(obs1, true)))
print(c1 | c2)
cat(sprintf("ceiling at sigma = 0.3: %.2f | at sigma = 0.7: %.2f\n",
            1 - 0.3^2 / var_obs, 1 - 0.7^2 / var_obs))



<p class="caption"><b>Figure 3.</b> Left: the maximum achievable R² as a function of the assumed measurement noise, for this dataset's observed variance of 1.23. At our estimated σ = 0.51 the ceiling is 0.79; at the optimistic σ = 0.3 it would be 0.93, at a pessimistic σ = 0.7 only 0.60. The curve is steep, which is why the uncertainty estimate deserves the care Section 4 gave it. Right: what a perfect model looks like when plotted against noisy observations — a cloud around the diagonal whose R² on this simulated draw is 0.795, as the analytic ceiling predicts. If your model's scatter plot looks tighter than this on data of this kind, look for the leak before celebrating.</p>

<div class="exercise"><h4>Exercise 2</h4>
<p>Restrict the uncertainty estimate to compounds measured in at least three documents and recompute σ. Does it go up or down, and why might selecting the better-studied compounds change the answer in either direction? Then recompute the ceiling using only IC50 measurements and state what you would report in a methods section.</p></div>

## 6. The activity threshold is a modelling decision, not a fact

Most published models on data like these are classifiers, and a classifier needs classes. The potency distribution is continuous, so the classes are made by cutting it at a threshold — usually pActivity ≥ 6 (1 µM), 6.5 or 7 (100 nM) — and the threshold is chosen by convention rather than by biology. A sensitivity analysis is therefore not optional: it shows the reader how much of the result is the threshold.



In [ ]:
ex_c <- comp |> filter(!censored)
sens <- lapply(c(6.0, 6.5, 7.0), function(t) {
  act <- ex_c |> filter(pActivity >= t)
  sc <- sort(table(act$scaffold_smiles), decreasing = TRUE)
  data.frame(threshold = t, n_active = nrow(act), n_inactive = nrow(ex_c) - nrow(act),
             ratio = nrow(act) / (nrow(ex_c) - nrow(act)),
             scaffolds_in_actives = length(sc), singleton_scaffolds = sum(sc == 1),
             top_scaffold_share = 100 * sc[[1]] / nrow(act))
}) |> do.call(what = rbind)
print(sens, row.names = FALSE, digits = 3)
switch_n <- sum((ex_c$pActivity >= 6.0) != (ex_c$pActivity >= 7.0))
cat(sprintf("\ncompounds that change class between the 6.0 and 7.0 thresholds: %d (%.0f%% of the modelling set)\n",
            switch_n, 100 * switch_n / nrow(ex_c)))
cat(sprintf("compounds within one measurement sigma of the 6.5 threshold: %d (%.0f%%)\n",
            sum(abs(ex_c$pActivity - 6.5) < sigma_hat),
            100 * mean(abs(ex_c$pActivity - 6.5) < sigma_hat)))
cat(sprintf("majority-class baseline accuracy: %.0f%% at 6.0, %.0f%% at 6.5, %.0f%% at 7.0\n",
            100 * max(sens$n_active[1], sens$n_inactive[1]) / nrow(ex_c),
            100 * max(sens$n_active[2], sens$n_inactive[2]) / nrow(ex_c),
            100 * max(sens$n_active[3], sens$n_inactive[3]) / nrow(ex_c)))


In [ ]:
bal <- sens |> select(threshold, active = n_active, inactive = n_inactive) |>
  pivot_longer(-threshold, names_to = "class", values_to = "n")
t1 <- ggplot(bal, aes(factor(threshold), n, fill = class)) +
  geom_col(position = position_dodge(0.75), width = 0.7) +
  geom_text(data = sens, inherit.aes = FALSE,
            aes(factor(threshold), pmax(n_active, n_inactive),
                label = sprintf("ratio %.2f", ratio)), vjust = -0.5, size = 2.4) +
  scale_fill_manual(values = c(active = unname(PAL["green"]),
                               inactive = unname(PAL["orange"])), name = NULL) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  labs(x = "pActivity threshold", y = "compounds",
       title = "Class balance flips between 6.0 and 6.5") +
  theme(legend.position = "bottom")
sc_long <- sens |> select(threshold, `distinct scaffolds` = scaffolds_in_actives,
                          `of which singletons` = singleton_scaffolds) |>
  pivot_longer(-threshold, names_to = "what", values_to = "n")
t2 <- ggplot(sc_long, aes(factor(threshold), n, fill = what)) +
  geom_col(position = position_dodge(0.75), width = 0.7) +
  scale_fill_manual(values = unname(PAL[c("blue", "purple")]), name = NULL) +
  labs(x = "pActivity threshold", y = "scaffolds among the actives",
       title = "The active set narrows chemically") +
  theme(legend.position = "bottom")
t3 <- ggplot(ex_c, aes(pActivity)) +
  annotate("rect", xmin = 6.5 - sigma_hat, xmax = 6.5 + sigma_hat, ymin = 0, ymax = Inf,
           fill = unname(PAL["orange"]), alpha = 0.12) +
  geom_histogram(binwidth = 0.15, boundary = 3, fill = "grey80") +
  geom_vline(xintercept = c(6.0, 6.5, 7.0),
             colour = unname(PAL[c("green", "blue", "orange")]), linewidth = 0.5) +
  labs(x = "pActivity", y = "compounds",
       title = "Where the cuts fall (+-1 sigma shaded)")
print(t1 | t2 | t3)



<p class="caption"><b>Figure 4.</b> Threshold sensitivity. Left: the active-to-inactive ratio goes from 2.19 at pActivity ≥ 6.0 (a majority-active set) through 0.91 at 6.5 (almost balanced) to 0.49 at 7.0 (a two-to-one majority-inactive set) — three different learning problems from one dataset. Middle: the actives also become chemically narrower, from 802 distinct scaffolds at 6.0 to 366 at 7.0, while the single largest scaffold family grows from 10.2% to 16.1% of the actives. Right: the three cuts on the potency distribution, with the ±1σ measurement-uncertainty band around 6.5 shaded — 1,011 compounds, 37% of the modelling set, lie inside that band and could fall on either side of the label on a repeat experiment.</p>

One footnote, because it is a lesson in itself. The reference table shipped with the course stores `pActivity` rounded to three decimals, and this document works with the unrounded value aggregated by its own protocol; 23 compounds sit at exactly 6.500 after rounding and just below it before. At the 6.5 threshold that difference alone moves the counts from 1,316/1,448 (here) to 1,339/1,425 (the reference README). A rounding convention changed 1.7% of the labels — which is the whole argument of this section in miniature, and a good reason to carry the unrounded value through a pipeline and round only for display.

Two conclusions follow, and both belong in a methods section. First, report the threshold sensitivity, because a classifier's accuracy is not comparable across thresholds — a majority-class baseline alone moves from 69% at 6.0 to 67% at 7.0. Second, remember that 984 compounds, 36% of the set, change class between 6.0 and 7.0; the labels are a coarse summary of a continuous measurement, and the regression target is almost always the more honest choice.<sup class="c"><a href="#r19">[19]</a></sup><sup class="c"><a href="#r20">[20]</a></sup>

## 7. Bias diagnostics

A dataset assembled from the medicinal-chemistry literature is not a sample of chemical space; it is the record of what people chose to make. We measure that with the **Bemis–Murcko scaffold** — the molecule stripped to its ring systems and their linkers, which is how chemists name a compound family<sup class="c"><a href="#r14">[14]</a></sup> — and with the source document. Three concentrations follow, and each one breaks a different downstream assumption.



In [ ]:
gini <- function(counts) {
  x <- sort(as.numeric(counts)); n <- length(x)
  sum((2 * seq_len(n) - n - 1) * x) / (n * sum(x))
}
scaf <- sort(table(comp$scaffold_smiles), decreasing = TRUE)
docs <- sort(table(measurements$document_chembl_id), decreasing = TRUE)
cat(sprintf("scaffolds: %d distinct, %d singletons (%.0f%%), Gini %.3f\n",
            length(scaf), sum(scaf == 1), 100 * mean(scaf == 1), gini(scaf)))
cat(sprintf("largest scaffold family: %d compounds (%.1f%% of the table); ten largest = %.1f%%\n",
            scaf[[1]], 100 * scaf[[1]] / nrow(comp), 100 * sum(scaf[1:10]) / nrow(comp)))
cat("scaffolds needed to cover half the compounds:",
    which(cumsum(scaf) / nrow(comp) >= 0.5)[1], "\n")
cat(sprintf("\ndocuments: %d, Gini %.3f; largest contributes %.1f%% of all measurements; %d cover half the data\n",
            length(docs), gini(docs), 100 * docs[[1]] / nrow(measurements),
            which(cumsum(docs) / nrow(measurements) >= 0.5)[1]))

lorenz <- function(counts, name) {
  x <- sort(as.numeric(counts))
  data.frame(frac = seq_along(x) / length(x), cum = cumsum(x) / sum(x), what = name)
}
lz <- rbind(lorenz(scaf, sprintf("scaffolds (Gini %.2f)", gini(scaf))),
            lorenz(docs, sprintf("documents (Gini %.2f)", gini(docs))))
g1 <- ggplot(filter(lz, str_starts(what, "scaffolds")), aes(frac, cum)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", linewidth = 0.3) +
  geom_line(colour = PAL["blue"], linewidth = 0.7) +
  labs(x = "fraction of scaffolds (smallest first)", y = "fraction of the data",
       title = sprintf("Concentration across scaffolds (Gini %.2f)", gini(scaf)))
g2 <- ggplot(filter(lz, str_starts(what, "documents")), aes(frac, cum)) +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed", linewidth = 0.3) +
  geom_line(colour = PAL["orange"], linewidth = 0.7) +
  labs(x = "fraction of documents (smallest first)", y = "fraction of the data",
       title = sprintf("Concentration across documents (Gini %.2f)", gini(docs)))
g3 <- ggplot(data.frame(rank = 1:10, n = as.numeric(scaf[1:10])), aes(rank, n)) +
  geom_col(fill = PAL["purple"], width = 0.7) +
  scale_x_continuous(breaks = 1:10) +
  annotate("text", x = 5.5, y = max(scaf) * 0.85, size = 2.5,
           label = sprintf("top 10 = %.0f%% of the table", 100 * sum(scaf[1:10]) / nrow(comp))) +
  labs(x = "scaffold family (rank)", y = "compounds", title = "The ten largest chemical families")
print(g1 | g2 | g3)



<p class="caption"><b>Figure 5.</b> How unevenly the data are distributed. Left: the Lorenz curve across scaffold families — Gini 0.47, with 1,453 of 2,005 scaffolds (72%) represented by a single compound while the largest family holds 214. Middle: the same curve across source documents, far more unequal at Gini 0.82 — seven of 222 documents supply half of all measurements and the largest alone supplies 20.2%. Right: the ten largest families, together 13.5% of the table. A dataset with this shape is not a sample; it is a handful of optimisation campaigns plus a long tail of one-offs.</p>

Each concentration has a specific consequence.

- **Analogue bias.** When a few families dominate, a model can score well by memorising the families rather than learning chemistry, and a random train/test split hands it the answer. A **scaffold split**, which assigns whole families to one side, is the standard correction and typically drops reported performance sharply — which is the point.<sup class="c"><a href="#r16">[16]</a></sup><sup class="c"><a href="#r17">[17]</a></sup>
- **Source bias.** When seven documents supply half the measurements, the dataset inherits seven laboratories' assay conditions, compound preferences and reporting conventions. The 54% of potency variance attributable to the assay (Unit 4) is the measurable part of this. It is the chemical analogue of the **batch effect** in high-throughput biology, where the processing run leaves a signature larger than the biological signal<sup class="c"><a href="#r25">[25]</a></sup> and has to be removed explicitly, for instance with empirical-Bayes adjustment.<sup class="c"><a href="#r35">[35]</a></sup>
- **Singleton scaffolds.** The 1,453 families with one member are the compounds most unlike anything else; they are where a model's applicability domain ends (Unit 17), and they are also where a scaffold split puts most of its test set.



In [ ]:
set.seed(0)
idx <- sample(nrow(ex_c)); n_test <- floor(0.2 * nrow(ex_c))
rand_test <- idx[seq_len(n_test)]; rand_train <- idx[-seq_len(n_test)]
sv <- ex_c$scaffold_smiles
shared_random <- mean(sv[rand_test] %in% unique(sv[rand_train]))

set.seed(0)
groups <- table(ex_c$scaffold_smiles)
groups <- groups[sample(length(groups))]          # shuffle whole families, then fill the test set
csum <- cumsum(as.numeric(groups))
test_scaffolds <- names(groups)[seq_len(which(csum >= n_test)[1])]
scaf_mask <- ex_c$scaffold_smiles %in% test_scaffolds
shared_scaffold <- mean(sv[scaf_mask] %in% unique(sv[!scaf_mask]))

cat(sprintf("random 80/20 split : test compounds whose scaffold is also in train: %.0f%%\n",
            100 * shared_random))
cat(sprintf("scaffold split     : test compounds whose scaffold is also in train: %.0f%%\n",
            100 * shared_scaffold))
cat(sprintf("test-set size: random %d | scaffold %d compounds\n", n_test, sum(scaf_mask)))
cat("\nclass balance (pActivity >= 6.5) in each split:\n")
cat(sprintf("  random   : train %.2f | test %.2f\n",
            mean(ex_c$pActivity[rand_train] >= 6.5), mean(ex_c$pActivity[rand_test] >= 6.5)))
cat(sprintf("  scaffold : train %.2f | test %.2f\n",
            mean(ex_c$pActivity[!scaf_mask] >= 6.5), mean(ex_c$pActivity[scaf_mask] >= 6.5)))



This is the experiment that exhibits the failure mode. A random split leaves 65% of the test compounds with a relative in the training set; a scaffold split, by construction, leaves none. The second number also shows the hidden cost: because whole families move together, the class balance of the two sides drifts apart — from 0.48/0.47 under the random split to 0.50/0.40 under the scaffold split on this draw — so a scaffold split is harder *and* noisier, and comparing a scaffold-split score with a random-split score from another paper is meaningless. Report which split you used, and prefer the scaffold or time split when the question is prospective.<sup class="c"><a href="#r15">[15]</a></sup><sup class="c"><a href="#r19">[19]</a></sup> The same failure mode in structure-based benchmarks is why decoy sets have had to be rebuilt more than once.<sup class="c"><a href="#r18">[18]</a></sup>

One more quality screen belongs here. Some molecules are active in many unrelated assays not because they bind anything but because they aggregate, react or fluoresce; Baell and Holloway catalogued the substructures responsible as **PAINS** filters.<sup class="c"><a href="#r12">[12]</a></sup> Related rule sets exist for reactive and promiscuous chemistry.<sup class="c"><a href="#r13">[13]</a></sup> Treat a hit as a flag for human review, never as an automatic exclusion — the filters have well-documented false positives, and several marketed drugs match them.

<div class="box"><h4>Substitution in the R edition — the PAINS screen</h4>
<p>A PAINS screen is a <strong>SMARTS substructure search</strong>, and R has no cheminformatics toolkit in this environment to run one: RDKit's <code>FilterCatalog</code>, which the Python edition uses, has no R binding, and <code>ChemmineR</code>'s SMARTS search requires the OpenBabel bindings. The screen is therefore shown below as complete, non-evaluated source for a reader who has RDKit available (through <code>reticulate</code> or a Python script), and the quantitative demonstration is carried out instead with a <strong>property-based alert</strong> that R can compute from the descriptor columns already in the table: the count of Lipinski rule-of-five violations.<sup class="c"><a href="#r36">[36]</a></sup> It is a different screen answering a different chemical question, and it is used here only to make the same methodological point — flag, record, review, do not silently delete.</p></div>



In [ ]:
library(reticulate)   ## the PAINS screen as RDKit runs it; not evaluated here (no SMARTS engine)
rdkit  <- import("rdkit.Chem")
filters <- import("rdkit.Chem.FilterCatalog")
params <- filters$FilterCatalogParams()
params$AddCatalog(filters$FilterCatalogParams$FilterCatalogs$PAINS)
catalog <- filters$FilterCatalog(params)

pains_alert <- function(smiles) {
  mol <- rdkit$MolFromSmiles(smiles)
  if (is.null(mol)) return(NA_character_)
  entry <- catalog$GetFirstMatch(mol)
  if (is.null(entry)) NA_character_ else entry$GetDescription()
}
comp$pains <- vapply(comp$std_smiles, pains_alert, character(1))
table(comp$pains, useNA = "ifany")


In [ ]:
comp <- comp |>
  mutate(ro5_violations = (MolWt > 500) + (MolLogP > 5) + (HBD > 5) + (HBA > 10),
         alert = ro5_violations >= 2)
cat("rule-of-five violations per compound:\n"); print(table(comp$ro5_violations))
cat(sprintf("\nflagged (2 or more violations): %d of %d compounds (%.1f%%)\n",
            sum(comp$alert), nrow(comp), 100 * mean(comp$alert)))
cat(sprintf("which criterion fires: MolWt>500 %d | MolLogP>5 %d | HBD>5 %d | HBA>10 %d\n",
            sum(comp$MolWt > 500), sum(comp$MolLogP > 5), sum(comp$HBD > 5), sum(comp$HBA > 10)))
cat(sprintf("median pActivity  flagged %.2f  vs  unflagged %.2f\n",
            median(comp$pActivity[comp$alert]), median(comp$pActivity[!comp$alert])))
cat(sprintf("censored-only share  flagged %.0f%%  vs  unflagged %.0f%%\n",
            100 * mean(comp$censored[comp$alert]), 100 * mean(comp$censored[!comp$alert])))

crit <- data.frame(criterion = c("MolWt > 500", "MolLogP > 5", "HBA > 10", "HBD > 5"),
                   n = c(sum(comp$MolWt > 500), sum(comp$MolLogP > 5),
                         sum(comp$HBA > 10), sum(comp$HBD > 5)))
a1 <- ggplot(crit, aes(n, reorder(criterion, n))) +
  geom_col(fill = PAL["orange"], width = 0.65) +
  geom_text(aes(label = n), hjust = -0.2, size = 2.4) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.18))) +
  labs(x = "compounds", y = NULL,
       title = sprintf("Property alerts (%d compounds flagged)", sum(comp$alert)))
a2 <- ggplot(comp, aes(pActivity,
                       colour = ifelse(alert, "property alert", "no alert"))) +
  stat_ecdf(linewidth = 0.6) +
  scale_colour_manual(values = c(`no alert` = unname(PAL["blue"]),
                                 `property alert` = unname(PAL["orange"])), name = NULL) +
  labs(y = "cumulative fraction", title = "Flagged compounds are not more potent") +
  theme(legend.position = "bottom")
print(a1 | a2)



<p class="caption"><b>Figure 6.</b> Left: which rule-of-five criterion fires, over the whole table — lipophilicity (552 compounds above logP 5) and size (391 above 500 Da) dominate, while hydrogen-bond counts almost never do; 210 compounds (4.9%) violate two or more criteria and are flagged. Right: the potency ECDFs of flagged and unflagged compounds. The flagged set is <em>weaker</em> (median pActivity 5.28 against 6.00) and far more often censored (60% against 34%), so these greasy, heavy molecules are not the dataset's potent compounds. As with a PAINS hit, the right action is to record the flag in the table, inspect the flagged compounds if they turn out to be influential, and report the count — not to delete them, which would remove exactly the part of the potency range a model most needs to learn.</p>

## 8. Responsible data: when the rows are people

Everything so far concerned molecules, which have no rights. The moment a dataset derives from human participants the obligations change completely, and they are legal as well as ethical. This section uses the clinical metadata of a public gene-expression series — a study of gingival tissue in periodontitis, deposited in NCBI GEO<sup class="c"><a href="#r26">[26]</a></sup><sup class="c"><a href="#r27">[27]</a></sup> — to show what a properly minimised human dataset looks like. We inspect the **schema** and nothing else.

<div class="box bridge"><h4>Domain context — the vocabulary of data protection</h4>
<p>A <strong>direct identifier</strong> names a person: name, national identity number, medical-record number, e-mail. A <strong>quasi-identifier</strong> does not name anyone alone but can when combined with other quasi-identifiers or an external dataset — postcode, date of birth, sex, admission date, rare diagnosis. A <strong>sensitive attribute</strong> is the information the study is about and that the participant would not want disclosed: a diagnosis, a genotype, a behaviour. <strong>Pseudonymisation</strong> replaces direct identifiers with a code while a re-identification key is kept elsewhere; the data remain personal data in law because the link still exists. <strong>Anonymisation</strong> removes the possibility of re-identification altogether, which is much harder than it sounds and is a property of the data plus everything else in the world, not of the file. <strong>Data minimisation</strong> is the principle that you collect and keep only the fields the stated purpose requires — the single most effective protection, because a field that was never collected cannot leak.</p></div>



In [ ]:
meta <- load_data("geo_periodontitis_meta.csv")
schema <- data.frame(
  column    = names(meta),
  class     = vapply(meta, \(x) class(x)[1], character(1)),
  n_unique  = vapply(meta, \(x) length(unique(na.omit(x))), integer(1)),
  n_missing = vapply(meta, \(x) sum(is.na(x)), integer(1)),
  example   = vapply(meta, \(x) substr(as.character(na.omit(x)[1]), 1, 42), character(1)),
  row.names = NULL)
cat("rows:", nrow(meta), "| columns:", ncol(meta),
    "| distinct participants:", n_distinct(meta$patient_id), "\n")
print(schema, row.names = FALSE)
absent <- setdiff(c("age", "sex", "gender", "ethnicity", "smoking", "date_of_birth",
                    "postcode", "hospital", "sampling_date", "name"), names(meta))
cat("\nfields that are NOT present:", paste(absent, collapse = ", "), "\n")



The schema is the lesson. Eleven columns, 138 rows, 69 participants, and **no age, no sex, no ethnicity, no smoking status, no date, no site of recruitment** — none of the quasi-identifiers that make health data re-identifiable. `patient_id` is a sequential pseudonym (`P001`…) with no key distributed alongside the file; the only attributes present are the phenotype the study is about (diseased or healthy gingival site), the tissue, and the sampling structure. A reader can reproduce the paired analysis and cannot learn anything about a person.

That restraint has a cost — age, sex and smoking are genuine confounders for periodontitis, and their absence limits what the data can support, a limitation the depositors accepted deliberately. It also has a documented justification: re-identification attacks on health data succeed routinely when quasi-identifiers are present,<sup class="c"><a href="#r29">[29]</a></sup> and with fifteen demographic attributes it is possible to re-identify almost any individual in an incomplete sample.<sup class="c"><a href="#r30">[30]</a></sup>

The standard formal criterion is **k-anonymity**: a release is *k*-anonymous if every combination of quasi-identifiers appearing in it is shared by at least *k* records, so no record is distinguishable from fewer than *k*−1 others.<sup class="c"><a href="#r28">[28]</a></sup> We demonstrate it on a **synthetic** clinical table generated in this document — the real GEO file has no quasi-identifiers to generalise, which is the point we just made.



In [ ]:
set.seed(7)
N <- 400
synth <- data.frame(
  record_id = sprintf("S%04d", seq_len(N) - 1),
  age       = sample(19:80, N, replace = TRUE),
  sex       = sample(c("F", "M"), N, replace = TRUE),
  region    = sample(sprintf("R%02d", 1:12), N, replace = TRUE),
  diagnosis = sample(c("periodontitis", "gingivitis", "healthy"), N, replace = TRUE,
                     prob = c(.5, .3, .2)))
QI <- c("age", "sex", "region")
class_sizes <- function(df, qi) as.numeric(table(do.call(paste, c(df[qi], sep = "|"))))
k_profile <- function(df, qi) {
  s <- class_sizes(df, qi)
  c(k = min(s), unique = sum(s == 1), below_5 = sum(s < 5), classes = length(s))
}
gen <- synth |>
  mutate(age = cut(age, c(18, 30, 45, 60, 100), labels = c("19-30", "31-45", "46-60", "61+")),
         region = paste0(substr(region, 1, 2),
                         ifelse(as.integer(substr(region, 3, 3)) <= 6, "north", "south")))
p0 <- k_profile(synth, QI); p1 <- k_profile(gen, QI)
cat(sprintf("raw quasi-identifiers (exact age, sex, region): k = %d, %d unique records, %d groups smaller than 5, %d classes\n",
            p0[1], p0[2], p0[3], p0[4]))
cat(sprintf("after generalisation (age band, sex, region half): k = %d, %d unique records, %d groups smaller than 5, %d classes\n",
            p1[1], p1[2], p1[3], p1[4]))
cat(sprintf("\ndata minimisation instead: dropping 'region' entirely gives k = %d\n",
            k_profile(gen, c("age", "sex"))[1]))
ld <- gen |> group_by(age, sex, region) |> summarise(l = n_distinct(diagnosis), .groups = "drop")
cat(sprintf("l-diversity of the generalised table (distinct diagnoses per group): min %d, max %d\n",
            min(ld$l), max(ld$l)))


In [ ]:
s0 <- class_sizes(synth, QI); s1 <- class_sizes(gen, QI)
k1 <- ggplot(data.frame(size = s0), aes(size)) +
  geom_histogram(binwidth = 1, fill = PAL["orange"], colour = "white", linewidth = 0.2) +
  labs(x = "records sharing a quasi-identifier combination", y = "equivalence classes",
       title = sprintf("Raw: k = %d", p0[1]))
k2 <- ggplot(data.frame(size = s1), aes(size)) +
  geom_histogram(binwidth = 1, fill = PAL["green"], colour = "white", linewidth = 0.2) +
  geom_vline(xintercept = 5, linetype = "dashed", colour = PAL["orange"], linewidth = 0.4) +
  labs(x = "records sharing a quasi-identifier combination", y = NULL,
       title = sprintf("Generalised: k = %d  (dashed: k = 5)", p1[1]))
k3 <- ggplot(schema, aes(n_unique, factor(column, levels = rev(column)))) +
  geom_col(fill = PAL["blue"], width = 0.7) +
  scale_x_log10() +
  labs(x = "distinct values (log scale)", y = NULL,
       title = "GEO metadata: 11 fields, no demographics") +
  theme(axis.text.y = element_text(size = 6))
print(k1 | k2 | k3)



<p class="caption"><b>Figure 7.</b> Left: with exact age, sex and region, the synthetic table of 400 records is almost entirely unique — k = 1, with 311 of its 353 equivalence classes holding a single record (and every class smaller than five), so publishing it would be publishing individuals. Middle: after generalising age into four bands and region into two halves there are only 24 classes, the smallest with 6 members, so every class clears the conventional k = 5 line — at the cost of resolution in exactly the variables an epidemiologist wants. Right: the real GEO metadata for comparison — eleven fields, of which only the sample and patient pseudonyms and the free-text title are high-cardinality; the phenotype fields take one or two values and no demographic field exists at all. Minimisation at collection time achieves what generalisation afterwards can only approximate.</p>

Three further points, briefly, because they are the ones students most often meet first in practice.

- **k-anonymity is necessary, not sufficient.** If every record in an equivalence class shares the same sensitive value, membership of the class discloses it; **l-diversity** requires at least *l* distinct sensitive values per class.<sup class="c"><a href="#r31">[31]</a></sup> For analyses that release statistics rather than records, **differential privacy** gives a formal guarantee by adding calibrated noise to the output.<sup class="c"><a href="#r32">[32]</a></sup>
- **The workflow comes before the data.** Human-subjects work requires a protocol approved by an institutional ethics committee (in Chile, a *Comité Ético Científico* accredited by the relevant health authority) before recruitment; informed consent describing the actual secondary uses; a data-management plan saying where the data live, who may access them and when they will be destroyed; and a data-use agreement for anything shared outside the institution. The Declaration of Helsinki is the reference framework.<sup class="c"><a href="#r33">[33]</a></sup> Working with an already-public, already-de-identified set such as this GEO series normally falls outside that process, but *only* if you do not attempt to re-identify or to link it to another source — which is itself usually a condition of the terms of use.
- **The legal frame.** In the European Union the GDPR governs personal data, with pseudonymised data still in scope and anonymised data out of it. In Chile, personal-data protection became a constitutional right in 2018; the long-standing Ley 19.628 has been replaced by a modernised framework (Ley 21.719, 2024) that creates a supervisory agency and raises the standard for health data substantially, with entry into force staged. Treat the specific obligations as something to check with your institution's data-protection officer for the current year rather than from any textbook, including this one.

<div class="box bridge"><h4>Domain context — metadata standards</h4>
<p>For biological data, the community answer to "what must be recorded?" is a <strong>minimum-information standard</strong>: MIAME for microarray experiments specified the protocol, array design, normalisation and sample annotation that must accompany any deposit,<sup class="c"><a href="#r34">[34]</a></sup> and its descendants cover sequencing, proteomics and imaging. These standards are why a public series such as GSE16134 can be reanalysed fifteen years later. They are the biological ancestor of the data card in the next section and of the FAIR principles<sup class="c"><a href="#r24">[24]</a></sup> that govern how the course's own frozen datasets are packaged.</p></div>



In [ ]:
per_patient <- as.data.frame(table(table(meta$patient_id)))
names(per_patient) <- c("samples", "participants")
m1 <- ggplot(per_patient, aes(samples, participants)) +
  geom_col(fill = PAL["blue"], width = 0.4) +
  geom_text(aes(label = participants), vjust = -0.4, size = 2.5) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  labs(x = "samples contributed by one participant", y = "participants",
       title = paste0(n_distinct(meta$patient_id), " participants, all paired"))
st <- as.data.frame(table(meta$status)); names(st) <- c("status", "n")
m2 <- ggplot(st, aes(status, n)) +
  geom_col(aes(fill = status), width = 0.5) +
  geom_text(aes(label = n), vjust = -0.4, size = 2.5) +
  scale_fill_manual(values = unname(PAL[c("orange", "green")]), guide = "none") +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  labs(x = NULL, y = "samples", title = "Balanced by design")
risk <- data.frame(
  kind = c("direct identifiers", "demographic quasi-identifiers", "date / location fields",
           "pseudonymous keys", "phenotype fields", "free-text fields"),
  n = c(0, 0, 0, 3, 7, 1))
m3 <- ggplot(risk, aes(n, factor(kind, levels = rev(kind)))) +
  geom_col(aes(fill = n == 0), width = 0.65) +
  scale_fill_manual(values = c(`TRUE` = unname(PAL["green"]), `FALSE` = "#f2c14e"),
                    guide = "none") +
  xlim(0, 6) +
  labs(x = "fields of this kind in the table", y = NULL, title = "Disclosure-risk inventory") +
  theme(axis.text.y = element_text(size = 6.5))
print(m1 | m2 | m3)
cat("samples per participant:", paste(per_patient$samples, per_patient$participants,
                                      sep = " -> ", collapse = "; "), "\n")
cat("the free-text field 'title' reproduces the sample index and site; it adds no information\n",
    "beyond the structured columns and would be the first field to drop under minimisation.\n")



<p class="caption"><b>Figure 8.</b> What a minimised human dataset still supports. Left: all 69 participants contribute exactly two samples, so the paired design — the statistically efficient one, since each participant is their own control — survives intact. Middle: 69 diseased and 69 healthy sites, balanced by construction. Right: a disclosure-risk inventory of the eleven fields: no direct identifiers, no demographic quasi-identifiers, no dates or locations; three pseudonymous keys (<code>sample_id</code>, <code>geo_accession</code>, <code>patient_id</code>), seven phenotype or design fields and one free-text title that merely restates the structured columns. This is the shape to aim for — everything the analysis needs, nothing that identifies anyone.</p>

## 9. The data card

A **data card** (or datasheet for a dataset) is a short structured document that travels with the data and answers the questions a reader would otherwise have to guess at: why it was made, how, what is in it, what is wrong with it, and what it must not be used for.<sup class="c"><a href="#r21">[21]</a></sup><sup class="c"><a href="#r22">[22]</a></sup><sup class="c"><a href="#r23">[23]</a></sup> Writing one takes twenty minutes and it is the single highest-return document in a data project. The chunk below generates ours from the objects computed in this document, so it cannot go stale.



In [ ]:
lg <- log_frame(cur)
card <- list(
  name = "ROCK1 bioactivity, curated compound-level table (R edition)",
  version = "ChEMBL_37 snapshot, curated by the protocol of this document",
  purpose = "training and benchmarking QSAR models of ROCK1 inhibition for the LBQC course",
  source = list(database = "ChEMBL release 37",
                target = "CHEMBL3231 (human ROCK1, UniProt Q13464)",
                licence = "CC BY-SA 3.0",
                query = "activity: standard_type in {IC50, Ki, Kd}"),
  size = list(raw_records = nrow(raw), records_after_curation = nrow(measurements),
              compounds = nrow(comp), records_excluded = nrow(raw) - nrow(measurements)),
  exclusions = setNames(
    lapply(which(lg$n_removed > 0 & startsWith(lg$code, "E")),
           \(i) list(reason = lg$description[i], n = lg$n_removed[i])),
    lg$code[lg$n_removed > 0 & startsWith(lg$code, "E")]),
  target_variable = list(name = "pActivity",
                         definition = "-log10(activity in mol/L), median over exact values",
                         mean = round(mean(comp$pActivity[!comp$censored]), 3),
                         sd = round(sy, 3),
                         range = round(range(comp$pActivity), 2)),
  censoring = list(compounds_with_exact_values = sum(!comp$censored),
                   censored_only = sum(comp$censored),
                   note = "censored compounds are retained and flagged; exclude them from regression targets explicitly"),
  measured_uncertainty = list(sigma_log_units = round(sigma_hat, 3),
                              method = "RMS inter-document replicate difference / sqrt(2)",
                              n_pairs = unc$n_pairs[1],
                              implied_R2_ceiling = round(var_true / var_obs, 3),
                              implied_RMSE_floor = round(sigma_hat, 3)),
  known_biases = list(distinct_scaffolds = length(scaf), singleton_scaffolds = sum(scaf == 1),
                      largest_scaffold_family = scaf[[1]], scaffold_gini = round(gini(scaf), 3),
                      document_gini = round(gini(docs), 3),
                      largest_document_share_pct = round(100 * docs[[1]] / nrow(measurements), 1),
                      property_alerts = sum(comp$alert),
                      pains_screen = "not run in the R edition (no SMARTS engine); see Section 7",
                      value_pile_ups = "several reported values are shared by >100 compounds (patent-style reporting)"),
  class_balance = setNames(lapply(seq_len(nrow(sens)),
                                  \(i) list(active = sens$n_active[i], inactive = sens$n_inactive[i])),
                           paste0("pActivity>=", sens$threshold)),
  recommended_use = c("scaffold split for prospective evaluation",
                      "regression on pActivity preferred over classification",
                      "report performance against the R2 ceiling, not against 0"),
  not_recommended = c(sprintf("random train/test split (%.0f%% scaffold overlap)",
                              100 * shared_random),
                      "silent deletion of censored compounds",
                      "comparison of accuracy across different activity thresholds"),
  human_subjects = "none; the dataset contains no personal data",
  generated = as.character(Sys.Date()))
cat(substr(toJSON(card, auto_unbox = TRUE, pretty = TRUE), 1, 2300))



<p class="caption"><b>Output (Section 9).</b> The data card as a machine-readable record. Every number in it was computed above, so regenerating the card after a change to the protocol updates the documentation automatically — the only way a data card stays true. In a repository it lives next to the CSV as <code>datacard.json</code> plus a human-readable <code>README.md</code>, and a dataset CHANGELOG records what changed between versions.</p>

The same information, as the table a reader actually reads:

| Field | Value |
|---|---|
| Dataset | ROCK1 bioactivity, compound level (ChEMBL_37 snapshot) |
| Rows / grain | 4,280 / one standardised parent structure (InChIKey) |
| Target | `pActivity` = −log<sub>10</sub>(activity in M), median of exact measurements |
| Provenance | 5,089 raw records → 5,045 after curation → 4,280 compounds; 44 excluded with reason codes |
| Measured uncertainty | σ ≈ 0.51 log units (780 inter-document replicate pairs) |
| Performance ceiling | R² ≤ 0.79, RMSE ≥ 0.51 log units |
| Censoring | 2,764 compounds with exact values, 1,516 censored-only (retained, flagged) |
| Chemical diversity | 2,005 scaffolds, 1,453 singletons, largest family 214 compounds, Gini 0.47 |
| Source concentration | 222 documents, Gini 0.82, largest supplies 20.2% of measurements |
| Quality flags | 210 rule-of-five property alerts (4.9%), recorded not removed; PAINS screen not run in the R edition; value pile-ups from patent-style reporting |
| Personal data | none |
| Licence | ChEMBL data CC BY-SA 3.0; cite Zdrazil *et al.* 2024<sup class="c"><a href="#r11">[11]</a></sup> |

<div class="box bridge"><h4>Bridge</h4>
<p><strong>Backwards:</strong> Unit 4 (<a href="../U4_Tidy_Data/index.html">Tidy data in the tidyverse</a>) built the aggregation this unit formalises, and Unit 5 (<a href="../U5_EDA/index.html">EDA and the grammar of graphics</a>) found the anomalies the data card now records; Unit 3 downloaded the raw snapshot and explains the ChEMBL schema (<a href="../U3_Getting_Data/index.html">Python edition</a>). <strong>Forwards:</strong> Units 11 and 12 consume this table and implement the scaffold split argued for in Section 7 (<a href="../U11_Supervised_Workflow/index.html">supervised workflow</a>, <a href="../U12_Validation/index.html">validation</a>); Unit 17 builds on the noise estimate of Section 4 (<a href="../U17_Uncertainty_AD/index.html">uncertainty and applicability domain</a>); Unit 21 analyses the periodontitis series whose metadata we inspected here (<a href="../U21_Omics/index.html">omics</a>); and Unit 23 turns the data card into the reporting standards a journal expects (<a href="../U23_Communication/index.html">scientific communication</a>). The <a href="../index.html">R-edition index</a> lists every unit and the <a href="../../Python/U6_Data_Quality/index.html">Python edition of this unit</a> runs the same protocol with RDKit. <strong>Sister courses:</strong> the chemistry of standardisation and curation is developed in <a href="../../../Cheminformatics/U4_Standardisation/index.html">Unit 4</a> and <a href="../../../Cheminformatics/U7_Data_Curation/index.html">Unit 7</a> of the cheminformatics course, scaffold-based chemical space in its <a href="../../../Cheminformatics/U8_Chemical_Space/index.html">Unit 8</a> and validation practice in its <a href="../../../Cheminformatics/U14_Validation/index.html">Unit 14</a>; the machine-learning consequences of a leaking split are covered in <a href="../../../BGM411/R/S1_Model_Evaluation/index.html">Model evaluation</a> of the AI course.</p></div>

## 10. Summary

- **Quality is fitness for a stated purpose**, and splits into correctness, precision, representativeness and documentation. Treat them separately; only the first is what most people mean by "cleaning".
- **Write the protocol as tested code with reason codes.** Four `testthat` tests guard the framework; the pipeline excluded 44 of 5,089 records (0.9%) — 3 without a structure, 13 carrying ChEMBL's validity flag (including one impossible IC50 of exactly 0 nM), 26 with no numeric value, 2 outside the heavy-atom window — and reproduced the reference table exactly: 4,280 compounds, identical censoring flags, identical scaffolds, potencies agreeing to 0.0005 log units.
- **Estimate the noise from your own replicates.** Seven hundred and eighty inter-document pairs give σ ≈ 0.506 log units, matching the published ≈0.5 for public bioactivity data; IC50 and Ki values for the same compound differ by −0.87 log units on average.
- **Convert the noise into a budget.** With σ = 0.506 and an observed spread of 1.108, the reliability is 0.792: no model can exceed R² ≈ 0.79 or beat RMSE ≈ 0.51 against these labels, and a perfect classifier agrees with a noisy label only 84–87% of the time.
- **The activity threshold is a decision.** Active-to-inactive goes 2.19 → 0.91 → 0.49 across 6.0 / 6.5 / 7.0; the active set narrows from 802 to 366 scaffolds; 984 compounds (36%) change class; 1,011 sit within one σ of the 6.5 cut. Report the sensitivity.
- **Diagnose the concentration.** 1,453 of 2,005 scaffolds are singletons (Gini 0.47), seven of 222 documents supply half the measurements (Gini 0.82), and a random split leaves 65% of test compounds with a relative in training against 0% for a scaffold split.
- **Flag, do not silently delete.** 210 property alerts (4.9%) are recorded in the table; they are weaker and more often censored than the rest, so deleting them would remove part of the potency range rather than clean it.
- **When the rows are people the rules change.** The GEO metadata we inspected carry no demographic quasi-identifiers at all — minimisation at collection beats generalisation afterwards — and k-anonymity, l-diversity, ethics approval, consent and the applicable data-protection law are prerequisites, not paperwork.
- **Ship a data card.** Generate it from the analysis so it cannot go stale.

## 11. Self-assessment

<details><summary>1. Why must structure standardisation precede de-duplication?</summary>
<p>Because the same molecule is deposited in several forms — free base, salt, hydrate, charged species — each with its own database identifier. De-duplicating on the identifier leaves those as separate compounds (ripasudil would appear three times in this dataset). Standardising first collapses them to one canonical parent whose InChIKey is the de-duplication key.</p></details>

<details><summary>2. A reviewer asks why you removed 13 records. What do you show them?</summary>
<p>The log line with its reason code (E03, "ChEMBL data_validity_comment flag"), the stored excluded rows, and the observation that twelve are potencies between 120 µM and 30 mM and one is an IC50 of exactly 0 nM whose logarithm is infinite. A curation step you cannot justify record by record is a step you should not have taken.</p></details>

<details><summary>3. How do you estimate experimental uncertainty without repeating any experiment?</summary>
<p>Use the replicates already in the data. For compounds measured in two <em>different</em> documents, take each document's mean and form the difference; if each is the truth plus independent noise of standard deviation σ, the differences have standard deviation σ√2, so σ = √(mean squared difference / 2). Here 780 pairs give σ ≈ 0.506 log units. Pairs from the same document are not independent and must be excluded.</p></details>

<details><summary>4. Your classifier reaches 96% accuracy on a public bioactivity benchmark. What do you check first?</summary>
<p>The split and the labels. With this dataset's noise, a perfect model agrees with the label only about 86% of the time, so 96% means either information has leaked between training and test (duplicate structures, salt forms, a random split with 65% scaffold overlap) or the test set is unrepresentative. Check de-duplication at InChIKey level first, then re-evaluate under a scaffold split.</p></details>

<details><summary>5. What exactly is the reliability of a dataset, and why is it the right baseline?</summary>
<p>Reliability is the fraction of the observed variance that is true signal rather than measurement noise: σ²<sub>true</sub>/σ²<sub>obs</sub> = 1 − σ²/σ²<sub>obs</sub>, here 0.792. It is the right baseline because a model is scored against noisy observations, so even a perfect predictor of the truth cannot explain the noise component. Reporting R² against 0 flatters the model; reporting it against the ceiling says how much of the available signal was captured.</p></details>

<details><summary>6. Why report threshold sensitivity instead of choosing the best threshold?</summary>
<p>Because the threshold creates the classes rather than discovering them: the active-to-inactive ratio moves from 2.19 to 0.49 across the three conventional cuts, and 36% of compounds change class. Choosing the threshold that maximises your metric is tuning on the outcome. Report all three, or model the continuous value.</p></details>

<details><summary>7. What is analogue bias, and which diagnostic reveals it?</summary>
<p>The dataset is dominated by families of close structural relatives, because it records optimisation campaigns rather than a sample of chemical space. The diagnostics are the scaffold count and its concentration — 2,005 scaffolds for 4,280 compounds, 72% singletons, Gini 0.47, largest family 214 — and the direct measurement of split leakage: 65% of a random test set has a scaffold relative in training.</p></details>

<details><summary>8. A scaffold split lowers your reported R² by 0.2. Is the model worse?</summary>
<p>No; the earlier number was wrong. The random split rewarded recognising known families, which is not the task. The scaffold-split number estimates performance on chemistry the model has not seen, which is what a prospective application needs. Note also that a scaffold split moves whole families and so disturbs the class balance between train and test, making the estimate noisier — report both the score and the split.</p></details>

<details><summary>9. Should compounds flagged by a structural or property alert be removed from a training set?</summary>
<p>Not automatically. PAINS filters flag substructures associated with assay interference but have well-documented false positives, and several approved drugs match them; a rule-of-five alert, like the one computed here, flags only that a molecule is large or greasy. Flag them in the table, report the count (210 here, 4.9%), inspect them if they are influential, and state what you did. In this dataset the flagged compounds are weaker and more often censored than the rest, so removing them would truncate the potency range.</p></details>

<details><summary>10. Distinguish a direct identifier, a quasi-identifier and a sensitive attribute, with an example of each.</summary>
<p>A direct identifier names the person (national identity number). A quasi-identifier identifies them only in combination or by linkage (date of birth, postcode, sex). A sensitive attribute is what the study is about and must not be disclosed (a periodontitis diagnosis, a genotype). Protection works on the quasi-identifiers: removing or generalising them is what makes the sensitive attribute safe to publish.</p></details>

<details><summary>11. The GEO metadata we examined have no age or sex column. Is that good practice or a limitation?</summary>
<p>Both, and the trade-off should be explicit. It removes the quasi-identifiers that drive re-identification risk, so the series can be public, but it also removes genuine confounders for periodontitis, so the data cannot support an adjusted analysis. The right response is to state the limitation in the data card rather than to wish the fields were there.</p></details>

<details><summary>12. What belongs in a data card that does not belong in a paper's methods section?</summary>
<p>The negative and operational information: what the dataset must <em>not</em> be used for, the known biases with numbers, the measured uncertainty and the performance ceiling it implies, the licence, the exclusion counts by reason code, and the version and date. A methods section describes what you did; a data card tells the next person what they can safely do.</p></details>

## Glossary

<dl class="gloss">
<dt>Analogue bias</dt><dd>Over-representation of families of close structural relatives, because the data record optimisation campaigns rather than a sample of chemical space.</dd>
<dt>Curation log</dt><dd>A per-step record of a cleaning pipeline: reason code, description, records removed, records remaining. The basis of a flow diagram and a data card.</dd>
<dt>Data card</dt><dd>A short structured document shipped with a dataset: purpose, provenance, composition, exclusions, known biases, recommended and discouraged uses.</dd>
<dt>Direct identifier / quasi-identifier / sensitive attribute</dt><dd>A field that names a person; a field that identifies them in combination or by linkage; the field the study is about and that must not be disclosed.</dd>
<dt>Gini coefficient</dt><dd>A measure of concentration between 0 (perfectly even) and 1 (all in one group); used here for scaffold families and source documents.</dd>
<dt>k-anonymity</dt><dd>A release in which every combination of quasi-identifiers is shared by at least k records, so no record is distinguishable from fewer than k−1 others.</dd>
<dt>Noise ceiling (reliability)</dt><dd>The fraction of observed variance that is true signal, 1 − σ²/σ²<sub>obs</sub>; the maximum R² any model can reach against those labels.</dd>
<dt>PAINS</dt><dd>Pan-assay interference compounds: substructures associated with apparent activity in many unrelated assays. A flag for review, not a deletion rule.</dd>
<dt>Pseudonymisation / anonymisation</dt><dd>Replacing identifiers with a code while a re-identification key exists (still personal data in law) versus removing the possibility of re-identification altogether.</dd>
<dt>Reason code</dt><dd>A short stable label attached to every excluded record, so any exclusion can be audited later.</dd>
<dt>Scaffold split</dt><dd>A train/test split that assigns whole Bemis–Murcko scaffold families to one side, so no test compound has a structural relative in training.</dd>
<dt>Threshold sensitivity</dt><dd>How class balance, composition and reported performance change when the activity cut-off is moved; a required part of any classification report on continuous data.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Fourches D, Muratov E, Tropsha A. Trust, but verify: on the importance of chemical structure curation in cheminformatics and QSAR modeling research. <i>Journal of Chemical Information and Modeling</i> 2010;50(7):1189–1204. doi:<a href="https://doi.org/10.1021/ci100176x">10.1021/ci100176x</a></li>
<li id="r2">Fourches D, Muratov E, Tropsha A. Trust, but verify II: a practical guide to chemogenomics data curation. <i>Journal of Chemical Information and Modeling</i> 2016;56(7):1243–1252. doi:<a href="https://doi.org/10.1021/acs.jcim.6b00129">10.1021/acs.jcim.6b00129</a></li>
<li id="r3">Kramer C, Kalliokoski T, Gedeck P, Vulpetti A. The experimental uncertainty of heterogeneous public Ki data. <i>Journal of Medicinal Chemistry</i> 2012;55(11):5165–5173. doi:<a href="https://doi.org/10.1021/jm300131x">10.1021/jm300131x</a></li>
<li id="r4">Kalliokoski T, Kramer C, Vulpetti A, Gedeck P. Comparability of mixed IC50 data — a statistical analysis. <i>PLoS ONE</i> 2013;8(4):e61007. doi:<a href="https://doi.org/10.1371/journal.pone.0061007">10.1371/journal.pone.0061007</a></li>
<li id="r5">Landrum GA, Riniker S. Combining IC50 or Ki values from different sources is a source of significant noise. <i>Journal of Chemical Information and Modeling</i> 2024;64(5):1560–1567. doi:<a href="https://doi.org/10.1021/acs.jcim.4c00049">10.1021/acs.jcim.4c00049</a></li>
<li id="r6">Tiikkainen P, Bellis L, Light Y, Franke L. Estimating error rates in bioactivity databases. <i>Journal of Chemical Information and Modeling</i> 2013;53(10):2499–2505. doi:<a href="https://doi.org/10.1021/ci400099q">10.1021/ci400099q</a></li>
<li id="r7">Williams AJ, Ekins S. A quality alert and call for improved curation of public chemistry databases. <i>Drug Discovery Today</i> 2011;16(17–18):747–750. doi:<a href="https://doi.org/10.1016/j.drudis.2011.07.007">10.1016/j.drudis.2011.07.007</a></li>
<li id="r8">Young D, Martin T, Venkatapathy R, Harten P. Are the chemical structures in your QSAR correct? <i>QSAR &amp; Combinatorial Science</i> 2008;27(11–12):1337–1345. doi:<a href="https://doi.org/10.1002/qsar.200810084">10.1002/qsar.200810084</a></li>
<li id="r9">Bento AP, Hersey A, Félix E, <i>et al.</i> An open source chemical structure curation pipeline using RDKit. <i>Journal of Cheminformatics</i> 2020;12:51. doi:<a href="https://doi.org/10.1186/s13321-020-00456-1">10.1186/s13321-020-00456-1</a></li>
<li id="r10">Heller SR, McNaught A, Pletnev I, Stein S, Tchekhovskoi D. InChI, the IUPAC International Chemical Identifier. <i>Journal of Cheminformatics</i> 2015;7:23. doi:<a href="https://doi.org/10.1186/s13321-015-0068-4">10.1186/s13321-015-0068-4</a></li>
<li id="r11">Zdrazil B, Felix E, Hunter F, <i>et al.</i> The ChEMBL Database in 2023. <i>Nucleic Acids Research</i> 2024;52(D1):D1180–D1192. doi:<a href="https://doi.org/10.1093/nar/gkad1004">10.1093/nar/gkad1004</a></li>
<li id="r12">Baell JB, Holloway GA. New substructure filters for removal of pan assay interference compounds (PAINS) from screening libraries and for their exclusion in bioassays. <i>Journal of Medicinal Chemistry</i> 2010;53(7):2719–2740. doi:<a href="https://doi.org/10.1021/jm901137j">10.1021/jm901137j</a></li>
<li id="r13">Bruns RF, Watson IA. Rules for identifying potentially reactive or promiscuous compounds. <i>Journal of Medicinal Chemistry</i> 2012;55(22):9763–9772. doi:<a href="https://doi.org/10.1021/jm301008n">10.1021/jm301008n</a></li>
<li id="r14">Bemis GW, Murcko MA. The properties of known drugs. 1. Molecular frameworks. <i>Journal of Medicinal Chemistry</i> 1996;39(15):2887–2893. doi:<a href="https://doi.org/10.1021/jm9602928">10.1021/jm9602928</a></li>
<li id="r15">Sheridan RP. Time-split cross-validation as a method for estimating the goodness of prospective prediction. <i>Journal of Chemical Information and Modeling</i> 2013;53(4):783–790. doi:<a href="https://doi.org/10.1021/ci400084k">10.1021/ci400084k</a></li>
<li id="r16">Wallach I, Heifets A. Most ligand-based classification benchmarks reward memorization rather than generalization. <i>Journal of Chemical Information and Modeling</i> 2018;58(5):916–932. doi:<a href="https://doi.org/10.1021/acs.jcim.7b00403">10.1021/acs.jcim.7b00403</a></li>
<li id="r17">Sieg J, Flachsenberg F, Rarey M. In need of bias control: evaluating chemical data for machine learning in structure-based virtual screening. <i>Journal of Chemical Information and Modeling</i> 2019;59(3):947–961. doi:<a href="https://doi.org/10.1021/acs.jcim.8b00712">10.1021/acs.jcim.8b00712</a></li>
<li id="r18">Mysinger MM, Carchia M, Irwin JJ, Shoichet BK. Directory of Useful Decoys, Enhanced (DUD-E): better ligands and decoys for better benchmarking. <i>Journal of Medicinal Chemistry</i> 2012;55(14):6582–6594. doi:<a href="https://doi.org/10.1021/jm300687e">10.1021/jm300687e</a></li>
<li id="r19">Tropsha A. Best practices for QSAR model development, validation, and exploitation. <i>Molecular Informatics</i> 2010;29(6–7):476–488. doi:<a href="https://doi.org/10.1002/minf.201000061">10.1002/minf.201000061</a></li>
<li id="r20">Cherkasov A, Muratov EN, Fourches D, <i>et al.</i> QSAR modeling: where have you been? Where are you going to? <i>Journal of Medicinal Chemistry</i> 2014;57(12):4977–5010. doi:<a href="https://doi.org/10.1021/jm4004285">10.1021/jm4004285</a></li>
<li id="r21">Gebru T, Morgenstern J, Vecchione B, <i>et al.</i> Datasheets for datasets. <i>Communications of the ACM</i> 2021;64(12):86–92. doi:<a href="https://doi.org/10.1145/3458723">10.1145/3458723</a></li>
<li id="r22">Mitchell M, Wu S, Zaldivar A, <i>et al.</i> Model cards for model reporting. <i>Proceedings of the Conference on Fairness, Accountability, and Transparency</i> 2019:220–229. doi:<a href="https://doi.org/10.1145/3287560.3287596">10.1145/3287560.3287596</a></li>
<li id="r23">Pushkarna M, Zaldivar A, Kjartansson O. Data cards: purposeful and transparent dataset documentation for responsible AI. <i>2022 ACM Conference on Fairness, Accountability, and Transparency</i> 2022:1776–1826. doi:<a href="https://doi.org/10.1145/3531146.3533231">10.1145/3531146.3533231</a></li>
<li id="r24">Wilkinson MD, Dumontier M, Aalbersberg IJ, <i>et al.</i> The FAIR Guiding Principles for scientific data management and stewardship. <i>Scientific Data</i> 2016;3:160018. doi:<a href="https://doi.org/10.1038/sdata.2016.18">10.1038/sdata.2016.18</a></li>
<li id="r25">Leek JT, Scharpf RB, Bravo HC, <i>et al.</i> Tackling the widespread and critical impact of batch effects in high-throughput data. <i>Nature Reviews Genetics</i> 2010;11:733–739. doi:<a href="https://doi.org/10.1038/nrg2825">10.1038/nrg2825</a></li>
<li id="r26">Barrett T, Wilhite SE, Ledoux P, <i>et al.</i> NCBI GEO: archive for functional genomics data sets — update. <i>Nucleic Acids Research</i> 2013;41(D1):D991–D995. doi:<a href="https://doi.org/10.1093/nar/gks1193">10.1093/nar/gks1193</a></li>
<li id="r27">Papapanou PN, Behle JH, Kebschull M, <i>et al.</i> Subgingival bacterial colonization profiles correlate with gingival tissue gene expression. <i>BMC Microbiology</i> 2009;9:221. doi:<a href="https://doi.org/10.1186/1471-2180-9-221">10.1186/1471-2180-9-221</a></li>
<li id="r28">Sweeney L. k-anonymity: a model for protecting privacy. <i>International Journal of Uncertainty, Fuzziness and Knowledge-Based Systems</i> 2002;10(5):557–570. doi:<a href="https://doi.org/10.1142/S0218488502001648">10.1142/S0218488502001648</a></li>
<li id="r29">El Emam K, Jonker E, Arbuckle L, Malin B. A systematic review of re-identification attacks on health data. <i>PLoS ONE</i> 2011;6(12):e28071. doi:<a href="https://doi.org/10.1371/journal.pone.0028071">10.1371/journal.pone.0028071</a></li>
<li id="r30">Rocher L, Hendrickx JM, de Montjoye Y-A. Estimating the success of re-identifications in incomplete datasets using generative models. <i>Nature Communications</i> 2019;10:3069. doi:<a href="https://doi.org/10.1038/s41467-019-10933-3">10.1038/s41467-019-10933-3</a></li>
<li id="r31">Machanavajjhala A, Kifer D, Gehrke J, Venkitasubramaniam M. l-diversity: privacy beyond k-anonymity. <i>ACM Transactions on Knowledge Discovery from Data</i> 2007;1(1):3. doi:<a href="https://doi.org/10.1145/1217299.1217302">10.1145/1217299.1217302</a></li>
<li id="r32">Dwork C, McSherry F, Nissim K, Smith A. Calibrating noise to sensitivity in private data analysis. <i>Theory of Cryptography (Lecture Notes in Computer Science 3876)</i> 2006:265–284. doi:<a href="https://doi.org/10.1007/11681878_14">10.1007/11681878_14</a></li>
<li id="r33">World Medical Association. WMA Declaration of Helsinki: ethical principles for medical research involving human subjects. <i>JAMA</i> 2013;310(20):2191–2194. doi:<a href="https://doi.org/10.1001/jama.2013.281053">10.1001/jama.2013.281053</a></li>
<li id="r34">Brazma A, Hingamp P, Quackenbush J, <i>et al.</i> Minimum information about a microarray experiment (MIAME) — toward standards for microarray data. <i>Nature Genetics</i> 2001;29:365–371. doi:<a href="https://doi.org/10.1038/ng1201-365">10.1038/ng1201-365</a></li>
<li id="r35">Johnson WE, Li C, Rabinovic A. Adjusting batch effects in microarray expression data using empirical Bayes methods. <i>Biostatistics</i> 2007;8(1):118–127. doi:<a href="https://doi.org/10.1093/biostatistics/kxj037">10.1093/biostatistics/kxj037</a></li>
<li id="r36">Lipinski CA, Lombardo F, Dominy BW, Feeney PJ. Experimental and computational approaches to estimate solubility and permeability in drug discovery and development settings. <i>Advanced Drug Delivery Reviews</i> 2001;46(1–3):3–26. doi:<a href="https://doi.org/10.1016/S0169-409X(00)00129-0">10.1016/S0169-409X(00)00129-0</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>The Turing Way Community. <i>The Turing Way: A Handbook for Reproducible, Ethical and Collaborative Research</i> — the "Research ethics" and "Research data management" chapters; free online.</li>
<li>El Emam K, Arbuckle L. <i>Anonymizing Health Data</i>. O'Reilly, 2013 — the practical companion to Section 8.</li>
<li>Wickham H, Bryan J. <i>R Packages</i>, 2nd ed. — how the <code>testthat</code> pattern of Section 2 becomes a package test suite; free online.</li>
<li>Hastie T, Tibshirani R, Friedman J. <i>The Elements of Statistical Learning</i>, 2nd ed. Springer, 2009 — the bias–variance framing behind the noise ceiling; free PDF.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>ChEMBL documentation on data curation and the <code>data_validity_comment</code> vocabulary (EMBL-EBI).</li>
<li><code>testthat</code> and <code>jsonlite</code> reference manuals; the <code>reticulate</code> vignette for calling RDKit from R.</li>
<li>Practical Cheminformatics (Pat Walters), posts on dataset curation, benchmark bias and "what your model is really learning".</li>
<li>European Data Protection Board guidelines on anonymisation and pseudonymisation; and, for Chile, the current guidance of the national data-protection authority and your institution's ethics committee.</li>
</ul>
